<a href="https://colab.research.google.com/github/keerthisingh-analyst/Uber-ride-analysis/blob/main/Uber_Ride_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load dataset
file_path = "https://gitlab.crio.do/me_notebook/me_jupyter_airbnbanalysis/-/raw/master/Airbnb_data.csv"
df = pd.read_csv(file_path)


In [ ]:
df.shape

Data Exploration


In [ ]:
df.info()

In [ ]:
df.head()

Handling Missing Values

In [ ]:
df.isnull().sum()

In [ ]:

# Fill missing values where necessary
df["reviews_per_month"].fillna(0, inplace=True)  # Replace NaNs with 0 for review counts
df.drop(columns=["last_review"], inplace=True)  # Drop last_review since it is not needed
# Replace only missing values in 'name' and 'host_name' with 'unknown'
df["name"].fillna("unknown", inplace=True)
df["host_name"].fillna("unknown", inplace=True)


In [ ]:
# Re-check missing values
df.isnull().sum()

Handling Outliers

In [ ]:


# Selecting only numeric columns
import matplotlib.pyplot as plt
numeric_columns = df.select_dtypes(include=['number']).columns


# plt.figure(figsize=(8, 5))
# plt.boxplot(df[price], vert=False, patch_artist=True)
# plt.title(f"Boxplot of price")
# plt.xlabel("Price")
# plt.grid(True)
# plt.show()

In [ ]:

# Ensure data types are correct
df["price"] = pd.to_numeric(df["price"], errors='coerce')
df["availability_365"] = pd.to_numeric(df["availability_365"], errors='coerce')


In [ ]:


# Remove outliers (if necessary)
df = df[df["price"] > 0]  # Remove listings with zero or negative price
df = df[df["minimum_nights"] < 365]  # Remove extreme long-term stays

In [ ]:



import numpy as np

Q1 = df["number_of_reviews"].quantile(0.25)  # First quartile
Q3 = df["number_of_reviews"].quantile(0.75)  # Third quartile
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


# Cap outliers at threshold values
df["number_of_reviews"] = np.where(df["number_of_reviews"] < lower_bound, lower_bound, df["number_of_reviews"])
df["number_of_reviews"] = np.where(df["number_of_reviews"] > upper_bound, upper_bound, df["number_of_reviews"])

In [ ]:
import numpy as np

Q1 = df["price"].quantile(0.25)  # First quartile
Q3 = df["price"].quantile(0.75)  # Third quartile
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


# Cap outliers at threshold values
df["price"] = np.where(df["price"] < lower_bound, lower_bound, df["price"])
df["price"] = np.where(df["price"] > upper_bound, upper_bound, df["price"])

In [ ]:
df['neighbourhood_group']

Understanding Customer preference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Understanding Customer Preferences


# TODO:
# 1. Group data by 'neighbourhood_group' and 'room_type', count listings, and reshape with unstack().
# 2. Fill any missing values with 0 for better visualization.
# 3. Create a bar plot to visualize room type popularity across neighborhoods:
#    - Set appropriate figure size.
#    - Use a colormap for visual appeal.
#    - Rotate x-axis labels for readability.
#    - Add title, axis labels, and legend.
#    - Display the plot.

grouped = df.groupby(['neighbourhood_group', 'room_type']).size().unstack()

grouped.plot(kind='bar')
plt.title("Most Popular Room Type Across Neighbourhoods")
plt.xlabel("Neighbourhood Group")
plt.ylabel("Count")
plt.show()


In [ ]:
df['neighbourhood']

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Understanding Customer Preferences

# TODO:
# 1. Create a new column 'price_range' by binning the 'price' into defined ranges with labels.
# 2. Group the data by 'neighbourhood_group' and 'price_range', then count listings and reshape using unstack().
# 3. Fill missing values with 0 for visualization consistency.
# 4. Generate a bar plot to show price range preferences across neighborhoods:
#    - Set figure size and use a distinct colormap.
#    - Rotate x-axis labels for clarity.
#    - Add title, axis labels, and legend.
#    - Display the plot.
bins = [0, 100, 500, 1000, 2000]
labels = ['Low', 'Medium', 'High', 'Large']
df['price_range'] = pd.cut(df['price'], bins=bins, labels=labels)

grouped1 = df.groupby(['neighbourhood_group', 'price_range']).size().unstack()
grouped1.plot(kind='bar')
plt.title("Customer Preferences for Price Ranges by Neighbourhood")
plt.xlabel("Neighbourhood Groups")
plt.ylabel("Count")
plt.legend(title='Price Range')
plt.show()




In [ ]:
bins = [0, 100, 500, 1000, 2000]
labels = ['Low', 'Medium', 'High', 'Large']

# Create price range column
df['price_range'] = pd.cut(df['price'], bins=bins, labels=labels)

# Group data
grouped1 = df.groupby(['neighbourhood_group', 'price_range']).size().unstack()

# Plot bar chart
grouped1.plot(kind='bar')

plt.title("Customer Preferences for Price Ranges by Neighbourhood")
plt.xlabel("Neighbourhood Groups")
plt.ylabel("Count")
plt.legend(title="Price Range")
plt.show()

In [ ]:

# TODO:
# 1. Group the data by 'neighbourhood' and sum the 'number_of_reviews' to find total reviews per area.
# 2. Sort the result in descending order to identify high-demand locations.
# 3. Create a bar plot for the top 15 neighborhoods with the most reviews:
#    - Set appropriate figure size and bar color.
#    - Rotate x-axis labels for readability.
#    - Add title, axis labels, and display the plot.

grouped2 = df.groupby("neighbourhood")['number_of_reviews'].sum()

top15 = grouped2.sort_values(ascending=False).head(15)

top15.plot(kind='bar')

plt.title("Top 15 Locations with Most Reviews (High Demand Areas)")
plt.xlabel("Neighbourhood Area")
plt.ylabel("Total Review Counts")
plt.xticks(rotation=45)
plt.show()

Pricing Strategy Analysis


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pricing Strategy Analysis

# TODO:
# 1. Group the data by 'neighbourhood_group' and calculate the average price.
# 2. Reset the index to prepare for plotting.
# 3. Create a bar plot showing average price per neighborhood group:
#    - Use a distinct color for the bars.
#    - Add a title and axis labels.
#    - Display the plot.

grouped3 = df.groupby("neighbourhood_group")["price"].mean().reset_index()

# 2. Create bar plot with distinct color
plt.figure(figsize=(8,5))
plt.bar(grouped3["neighbourhood_group"], grouped3["price"], color="orange")

# 3. Add title and labels
plt.title("Average Price by Neighbourhood Group")
plt.xlabel("Neighbourhood Groups")
plt.ylabel("Average Price")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
df['room_type']

In [ ]:
# TODO:
# 1. Group the data by 'room_type' and calculate the average price.
# 2. Reset the index to prepare the data for plotting.
# 3. Create a bar plot to show how average price varies by room type:
#    - Set figure size and choose a bar color.
#    - Add a title and axis labels.
#    - Display the plot.

grouped4 = df.groupby('room_type')['price'].mean().reset_index()

grouped4.plot(kind='bar', x="room_type")
plt.title("Average Price by Room Type")
plt.xlabel("Room Types")
plt.ylabel("Average Prices")
plt.show()

In [ ]:
# TODO:
# 1. Create a scatter plot to visualize the relationship between 'availability_365' and 'price'.
#    - Set figure size, point transparency (alpha), and color.
#    - Use seaborn's scatterplot for better aesthetics.
# 2. Add title and axis labels to clearly describe the plot.
# 3. Display the plot.

df.columns

In [ ]:
sns.scatterplot(data=df, x="availability_365", y="price")
plt.title("Relationship between Availability and Price")
plt.xlabel("availability_365")
plt.ylabel("price")
plt.show()

Growth Opportunity Analysis


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Market Competition & Growth Opportunities

# TODO:
# 1. Count the number of listings per 'neighbourhood_group' to assess market saturation.
# 2. Reset the index and rename columns for clarity.
# 3. Create a bar plot to visualize the number of listings per neighborhood group:
#    - Set figure size and choose a distinct bar color.
#    - Rotate x-axis labels for better readability.
#    - Add title and axis labels.
#    - Display the plot.

neighbour_list = df["neighbourhood_group"].value_counts()
print(neighbour_list)
neighbour_list.plot(kind='bar')
plt.title("Number of Listings per Neighbourhood (Market Saturation)")
plt.xlabel("Neighbourhood Groups")
plt.ylabel("Listing Count")
plt.show()

In [ ]:
# TODO:
# 1. Create a new column 'price_category' by binning the 'price' into defined categories (e.g., Budget, Luxury).
# 2. Count the number of listings in each price category to understand market competition.
# 3. Rename the resulting columns for clarity.
# 4. Create a bar plot to visualize the distribution of listings across price categories:
#    - Set figure size and bar color.
#    - Rotate x-axis labels for readability.
#    - Add title and axis labels.
#    - Display the plot.

bins = [0, 100, 300, 1000]
labels = ['Budget', 'Mid-Range', 'Luxury']

df['price_category'] = pd.cut(df['price'], bins=bins, labels=labels)

# 2️⃣ Count listings in each category
category_count = df['price_category'].value_counts().reset_index()

# 3️⃣ Rename columns
category_count.columns = ['Price Category', 'Listing Count']

print(category_count)

# 4️⃣ Create bar plot
plt.figure(figsize=(8,5))
plt.bar(category_count['Price Category'], category_count['Listing Count'], color='skyblue')

plt.xticks(rotation=45)
plt.title("Competition in Budget vs. Luxury Listings")
plt.xlabel("Price Category")
plt.ylabel("Number of Listings")

plt.show()


In [ ]:
# TODO:
# 1. Create a new column 'total_revenue' by multiplying 'price' with 'minimum_nights'.
# 2. Group data by 'neighbourhood' and calculate average total revenue to estimate potential earnings.
# 3. Sort the neighborhoods by revenue potential in descending order.
# 4. Create a bar plot for the top 15 neighborhoods with the highest average revenue:
#    - Set figure size and bar color.
#    - Rotate x-axis labels for readability.
#    - Add title and axis labels.
#    - Display the plot.
import matplotlib.pyplot as plt

# 1️⃣ Create total_revenue column
df['total_revenue'] = df['price'] * df['minimum_nights']

# 2️⃣ Group by neighbourhood and calculate average total revenue
grouped5 = df.groupby('neighbourhood')['total_revenue'].mean().reset_index()

# 3️⃣ Sort in descending order
grouped5 = grouped5.sort_values(by='total_revenue', ascending=False)

# 4️⃣ Take top 15
top15 = grouped5.head(15)

# 5️⃣ Plot
plt.figure(figsize=(12,6))
plt.bar(top15['neighbourhood'], top15['total_revenue'], color='orange')

plt.xticks(rotation=45)
plt.title("Top 15 Neighbourhoods with Highest Revenue Potential")
plt.xlabel("Neighbourhood")
plt.ylabel("Average Revenue")

plt.show()